# Challenge 1: Real-Time Weather Alerts Agent

# **1 | Install Dependencies**

In [ ]:
!pip install "google-adk[extensions]" google-cloud-aiplatform vertexai requests --quiet

# **2 | Imports and Configuration**

In [ ]:
import os
import getpass
import requests
from typing import Optional, List, Dict

PROJECT_ID = "qwiklabs-gcp-02-9791a279d0be"
LOCATION = "us-central1"

# TODO: Replace with your Google Maps Geocoding API key, or enter it at the prompt below.
# To obtain a key: APIs & Services -> Credentials -> Create Credentials -> API Key
# Then enable the Geocoding API: APIs & Services -> Library -> search "Geocoding API"
MAPS_API_KEY = getpass.getpass("Enter your Google Maps Geocoding API key: ")

ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key: ")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# Tell ADK to use Vertex AI credentials instead of a standalone Gemini API key
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

MODEL_GEMINI = "gemini-2.5-flash"

print("Configuration complete.")

# **3 | Initialize Vertex AI**

In [ ]:
import google.auth

credentials, project = google.auth.default()
print(f"Authenticated as project: {project or PROJECT_ID}")

# **4 | Tool: Get Latitude/Longitude from a Place Name**

In [ ]:
def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """
    Convert a city or place name to latitude and longitude using the
    Google Maps Geocoding API.

    Args:
        location (str): A city name or address (e.g., "Austin, TX").

    Returns:
        Optional[Dict[str, float]]: Dictionary with 'lat' and 'lon' keys,
        or None if the location could not be geocoded.
    """
    params = {"address": location, "key": MAPS_API_KEY}

    try:
        response = requests.get(
            "https://maps.googleapis.com/maps/api/geocode/json",
            params=params,
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()

        if data.get("status") == "OK" and data.get("results"):
            coords = data["results"][0]["geometry"]["location"]
            return {"lat": coords["lat"], "lon": coords["lng"]}

        print(f"Geocoding returned status: {data.get('status')}")
        return None

    except requests.RequestException as e:
        print(f"Geocoding error: {e}")
        return None


# Quick sanity check
print(get_lat_lon("New York, NY"))

# **5 | Tool: Get Extended Weather Forecast from NWS**

In [ ]:
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: List of forecast period dictionaries with
        'name', 'temperature', 'temperatureUnit', 'shortForecast', and 'detailedForecast'.
        Returns None if data is unavailable or an error occurs.
    """
    # NWS requires a descriptive User-Agent or requests may be rejected
    headers = {"User-Agent": "WeatherAlertAgent/1.0 (weather-agent@example.com)"}

    try:
        points_resp = requests.get(
            f"https://api.weather.gov/points/{lat:.4f},{lon:.4f}",
            headers=headers,
            timeout=10,
        )
        points_resp.raise_for_status()

        forecast_url = points_resp.json()["properties"]["forecast"]

        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()

        periods = forecast_resp.json()["properties"]["periods"]
        return [
            {
                "name": p["name"],
                "temperature": str(p["temperature"]),
                "temperatureUnit": p["temperatureUnit"],
                "shortForecast": p["shortForecast"],
                "detailedForecast": p["detailedForecast"],
            }
            for p in periods[:5]
        ]

    except Exception as e:
        print(f"Weather forecast error: {e}")
        return None


# Quick sanity check — Washington DC coordinates
print(get_extended_weather_forecast(38.8977, -77.0365))

# **6 | Agent Instructions**

In [ ]:
WEATHER_AGENT_INSTRUCTIONS = """
You are Pat, a friendly and knowledgeable real-time weather assistant for the United States.

Your capabilities:
- Look up the latitude and longitude for any US city using the get_lat_lon tool
- Fetch the extended weather forecast for that location using the get_extended_weather_forecast tool
- Provide a clear, helpful weather summary including current conditions and any notable alerts

How to respond:
1. Always use get_lat_lon first to resolve the city to coordinates
2. Then call get_extended_weather_forecast with those coordinates
3. Summarize the forecast in a friendly, easy-to-read format
4. Highlight any severe weather, extreme temperatures, or weather alerts
5. If the location is outside the United States, politely explain that you can only
   provide forecasts for US locations

Always be helpful, accurate, and concise.
"""

print("Agent instructions defined.")

# **7 | Build the Gemini Agent**

In [ ]:
from google.adk.agents import Agent

weather_agent_gemini = Agent(
    name="Pat_Gemini",
    model=MODEL_GEMINI,
    description="Pat the Friendly Weather Agent — powered by Gemini.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)

print("Gemini weather agent created.")

# **8 | Build the Third-Party Model Agent (Claude via LiteLLM)**

In [ ]:
from google.adk.models.lite_llm import LiteLlm

weather_agent_claude = Agent(
    name="Pat_Claude",
    model=LiteLlm(model="anthropic/claude-3-5-haiku-20241022"),
    description="Pat the Friendly Weather Agent — powered by Claude.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)

print("Claude weather agent created.")

# **9 | Helper: Run Agent**

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from IPython.display import Markdown, display

async def run_agent(agent, query: str, user_id: str = "test-user") -> str:
    """Run a query through the ADK Runner and return the final text response."""
    session_service = InMemorySessionService()
    runner = Runner(
        agent=agent,
        app_name=agent.name,
        session_service=session_service,
    )
    session = await session_service.create_session(
        app_name=agent.name,
        user_id=user_id,
    )
    content = types.Content(
        role="user",
        parts=[types.Part(text=query)],
    )
    response_text = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=content,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            response_text = event.content.parts[0].text
    return response_text or "No response received."

print("run_agent helper defined.")

# **10 | Test: Multiple US Cities with Gemini Agent**

In [ ]:
test_cities = [
    "What is the weather like in Chicago, IL?",
    "Give me a weather summary for Miami, FL.",
    "Are there any weather alerts for Phoenix, AZ?",
    "What should I expect weather-wise in Seattle, WA this week?",
]

print("=" * 60)
print("TESTING GEMINI AGENT — MULTIPLE US CITIES")
print("=" * 60)

for query in test_cities:
    print(f"\nQuery: {query}")
    print("-" * 40)
    response = await run_agent(weather_agent_gemini, query)
    display(Markdown(response))
    print()

# **11 | Test: Same Cities with Claude Agent**

In [ ]:
print("=" * 60)
print("TESTING CLAUDE AGENT — SAME CITIES")
print("=" * 60)

# Test a subset to keep runtime reasonable
test_subset = test_cities[:2]

for query in test_subset:
    print(f"\nQuery: {query}")
    print("-" * 40)
    response = await run_agent(weather_agent_claude, query)
    display(Markdown(response))
    print()

# **12 | Test: Edge Cases**

In [ ]:
print("=" * 60)
print("TESTING EDGE CASES")
print("=" * 60)

edge_cases = [
    "What's the weather in Denver, CO?",
    "What's the weather in Austin, Texas right now?",
    "What's the weather in London, England?",
]

for query in edge_cases:
    print(f"\nQuery: {query}")
    print("-" * 40)
    response = await run_agent(weather_agent_gemini, query)
    display(Markdown(response))
    print()

# **13 | Interactive Chat**

In [ ]:
import random

RANDOM_CITIES = [
    "New York, NY", "Los Angeles, CA", "Chicago, IL", "Houston, TX",
    "Phoenix, AZ", "Philadelphia, PA", "San Antonio, TX", "San Diego, CA",
    "Dallas, TX", "San Jose, CA", "Austin, TX", "Jacksonville, FL",
    "Fort Worth, TX", "Columbus, OH", "Indianapolis, IN", "Charlotte, NC",
    "San Francisco, CA", "Seattle, WA", "Denver, CO", "Nashville, TN",
    "Oklahoma City, OK", "El Paso, TX", "Washington, DC", "Las Vegas, NV",
    "Louisville, KY", "Memphis, TN", "Portland, OR", "Baltimore, MD",
    "Milwaukee, WI", "Albuquerque, NM", "Tucson, AZ", "Fresno, CA",
    "Sacramento, CA", "Kansas City, MO", "Mesa, AZ", "Atlanta, GA",
    "Omaha, NE", "Colorado Springs, CO", "Raleigh, NC", "Miami, FL",
    "Minneapolis, MN", "New Orleans, LA", "Cleveland, OH", "Honolulu, HI",
    "Anchorage, AK", "Boise, ID", "Salt Lake City, UT", "Baton Rouge, LA",
]

RANDOM_TRIGGERS = {"random", "surprise me", "surprise", "random city", "pick one", "you choose"}

def detect_agent_switch(text):
    lower = text.lower()
    switch_phrases = [
        "switch to claude", "use claude", "claude agent", "use the claude model",
        "switch to gemini", "use gemini", "gemini agent", "use the gemini model",
        "switch back to gemini", "go back to gemini",
    ]
    for phrase in switch_phrases:
        if phrase in lower:
            key = "claude" if "claude" in phrase else "gemini"
            cleaned = text.lower().replace(phrase, "").strip(" ,.-")
            return key, cleaned or None
    if lower.strip() in ("claude", "gemini"):
        return lower.strip(), None
    for key in ("claude", "gemini"):
        for prefix in (f"using {key}", f"with {key}", f"ask {key}"):
            if lower.startswith(prefix):
                cleaned = text[len(prefix):].strip(" ,")
                return key, cleaned or None
    return None, text


AGENTS = {"gemini": weather_agent_gemini, "claude": weather_agent_claude}
active_key = "gemini"

print("\u250c" + "\u2500" * 43 + "\u2510")
print("\u2502      Pat \u2014 Real-Time US Weather Chat      \u2502")
print("\u2514" + "\u2500" * 43 + "\u2518")
print(f"Active model : Gemini  (say 'use claude' to switch)")
print("Commands     : 'random' for a surprise city | 'quit' to end\n")

city = None

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nPat: Goodbye! Stay weather-aware!")
        break

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "q", "bye"):
        print("Pat: Goodbye! Stay weather-aware!")
        break

    if user_input.lower() in RANDOM_TRIGGERS:
        city = random.choice(RANDOM_CITIES)
        print(f"  [Random city: {city}]\n")
        query = f"What's the weather like in {city}?"
        agent_key = None
    else:
        agent_key, query = detect_agent_switch(user_input)

    if agent_key:
        if agent_key != active_key:
            active_key = agent_key
            print(f"  [Switched to {active_key.capitalize()}]")
        else:
            print(f"  [Already using {active_key.capitalize()}]")
        if not query:
            continue
    elif query is None:
        query = f"What's the weather like in {city}?"

    print(f"  [{active_key.capitalize()} thinking...]\n")
    response = await run_agent(AGENTS[active_key], query)
    display(Markdown(f"**Pat ({active_key.capitalize()}):** {response}"))
    print()